# Vivek Shah - CS 670 - Project 2 Part 1

This notebook is a guided runner for a digital version of the classic board game **Cluedo**. The game's source code lives in separate `.py` files in the `src` folder. Each section below explains one file, then runs a small piece of it. The last sections run the self-tests and start the game.

**How the game works:** someone has been murdered in the mansion. The game secretly picks **one character, one weapon and one room** and seals them in a confidential envelope. Players move around the mansion, enter rooms, and make **suggestions** ("I suggest it was Colonel Mustard, with the Revolver, in the Study"). The other players try to **refute** each suggestion by showing a matching card, and you work out what's in the envelope by elimination.

**Scope:** this is Part 1 only. A game runs until you type `QUIT`, which then reveals the envelope.

---

### Source Files

| File | What it contains | Part 1 requirement |
|---|---|---|
| `config.py` | Rule settings (die size, exact-roll rule, player limits) | — |
| `cards.py` | The 6 characters, 6 weapons and 9 rooms | Character and weapon definitions |
| `board_layout.py` | The mansion map, start squares, secret passages, directions | Mansion layout, starting positions |
| `board.py` | `Board` class: square queries, doors, route finding, map checks, drawing | Room connections, wall checks |
| `input_utils.py` | Input helpers and `validate_move` | Move validation, graceful input handling |
| `player.py` | `Player` and `HumanPlayer` (all keyboard prompts) | CLI prompts |
| `game.py` | `GameSetup` (envelope, dealing, tokens) and `CluedoGame` (turns, movement, suggestions, refutations) | Solution, dealing, turns, movement, suggestions |
| `tests.py` | Self-tests of the core rules | — |
| `main.py` | **Entry point:** player setup and `main()` | — |


### How to Run
* **In this notebook:** run the cells from top to bottom (**Runtime → Run all** in Colab). Then play in the **"Play the game"** cell (Section 8).
* **From a terminal:** `python main.py` in the `src` folder. The self-tests run with `python tests.py`.
* You can type these commands at **any** game prompt: `BOARD`, `NOTES`, `HAND`, `HELP`, `QUIT`.

## 1. Setup: Find the game files

This cell finds the folder that holds the `.py` files and makes Python import from it.

* **Jupyter on your computer:** the cell finds the `src` folder next to this notebook's folder.
* **Google Colab:** Colab starts empty, so the files have to be brought in first. Either:
  * Paste your GitHub repository URL into `REPO_URL` below, and the cell clones it.
  * Open the **Files** panel (the folder icon on the left), upload the nine `.py` files, and run this cell again.

`%autoreload` makes the notebook pick up any edits you make to the `.py` files without restarting.

In [ ]:
# Setup - locate the source files and make them importable
%load_ext autoreload
%autoreload 2

import os
import sys
import subprocess

# Only needed in Colab: the URL of the GitHub repository that holds the code.
REPO_URL = "https://github.com/VS-2K01/NJIT-CS670-Project2-Part1-Cluedo.git"


def find_source_dir(start="."):
  # Return the first folder under `start` that contains main.py and game.py, or None.
  for root, dirs, files in os.walk(start):
    # Skip hidden folders such as .git
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    if "main.py" in files and "game.py" in files:
      return os.path.abspath(root)
  return None


source_dir = find_source_dir()
if source_dir is None:
  # Local Jupyter: the notebook is in `Colab Demo`, and the code is in the sibling `src` folder.
  sibling = os.path.abspath(os.path.join("..", "src"))
  if os.path.isfile(os.path.join(sibling, "main.py")):
    source_dir = sibling
if source_dir is None and REPO_URL:
  # Download the repository
  subprocess.run(["git", "clone", REPO_URL], check=True)
  source_dir = find_source_dir()
if source_dir is None:
  raise FileNotFoundError(
    "Couldn't find the game's .py files. In Colab, set REPO_URL above, or upload the .py files with the Files panel (folder icon on the left), then run this cell again."
  )

# Work inside the source folder
os.chdir(source_dir)
if source_dir not in sys.path:
  # Let `import game` etc. find the files
  sys.path.insert(0, source_dir)

print("Using the game files in:", source_dir)
print("Files:", ", ".join(sorted(f for f in os.listdir(source_dir) if f.endswith(".py"))))

## 2. Settings and cards: `config.py` and `cards.py`

**`config.py`** holds the rule switches in one place:
* `DIE_SIDES = 6`: the game uses one standard six-sided die.
* `REQUIRE_EXACT_ROLL = True`: you must use your whole roll unless you enter a room.
* `MIN_PLAYERS = 2` and `MAX_PLAYERS = 6`.

**`cards.py`** defines the **21 cards**:
* **6 characters:** Miss Scarlett, Colonel Mustard, Mrs. White, Reverend Green, Mrs. Peacock, Professor Plum
* **6 weapons:** Candlestick, Dagger, Lead Pipe, Revolver, Rope, Wrench
* **9 rooms:** Kitchen, Ballroom, Conservatory, Dining Room, Billiard Room, Library, Lounge, Hall, Study

The `CHARACTERS` order is also the **turn order**, so Miss Scarlett always goes first. Two lookup tables are built from these lists:
* `CATEGORY_OF` takes a card and returns its category.
* `TOKEN_SYMBOL` gives each character a number from `1` to `6` for the map.

In [ ]:
import config
import cards

print(f"Die: d{config.DIE_SIDES}   exact roll required: {config.REQUIRE_EXACT_ROLL}   players: {config.MIN_PLAYERS}-{config.MAX_PLAYERS}")
print(f"{len(cards.CHARACTERS)} characters + {len(cards.WEAPONS)} weapons + {len(cards.ROOMS)} rooms = {len(cards.ALL_CARDS)} cards")
print("Characters (turn order):", ", ".join(cards.CHARACTERS))
print("Weapons:", ", ".join(cards.WEAPONS))
print("Rooms:", ", ".join(cards.ROOMS))

## 3. The mansion: `board_layout.py` and `board.py`

**`board_layout.py`** stores the mansion as a **grid** of **25 rows × 24 columns**, the same size as the classic board, with rooms in their usual places. The overview document allows a graph or a 2D grid. A grid fits the brief's "UP 2, RIGHT 2" moves.

| Symbol | Meaning |
|---|---|
| `.` | Hallway square (you can walk here) |
| `#` | Off the board / solid wall |
| `K B C D I L O H S` | Inside a room (Kitchen (K), Ballroom (B), Conservatory (C), Dining (D), Billiard (I), Library (L), Lounge (O), Hall (H), Study (S)) |
| lowercase letter (e.g. `k`) | A **door** into that room |
| `X` | The center of the board, where the envelope sits (you can't walk here) |

The file also defines:
* `START_POSITIONS`: each character's starting hallway square at the edge of the board.
* `SECRET_PASSAGES`: Kitchen ↔ Study and Conservatory ↔ Lounge.
* `DIRECTIONS`: UP, DOWN, LEFT and RIGHT only. There are no diagonal moves.

**`board.py`** has the `Board` class, which turns the text map into an object the game can ask questions about:
* **Square queries:** `is_hallway`, `is_door`, `room_at`, `neighbors`. Room squares act as walls, so **the only way into a room is through a door.**
* **Rooms:** `doors_of(room)` returns each door with its outside hallway square. `secret_passage_from(room)` returns the room at the other end of a passage.
* **`shortest_path`:** a breadth-first search that finds the nearest door of a goal room. It never cuts through a room and treats other tokens' squares as blocked. The route hints shown before each move use it.
* **`_validate`:** checks the map when the board is built. Every door must touch exactly one hallway square, every room needs a door, start squares must be hallways, and every room must be reachable from every start square.

Read squares as **(row, col)**: row numbers run down the left, column numbers across the top.

In [ ]:
from board import print_board_preview

print_board_preview()

## 4. Input and move checking: `input_utils.py`

This file holds the small tools behind the command-line interface:
* **`read_line`** wraps `input()`. Typing `QUIT`, closed input, or Colab's stop button raises `GameQuit`, which ends the game cleanly instead of crashing.
* **`parse_move_command`** turns `UP 2, RIGHT 3`, `u2 r3` or `up 2 right 3` into `[("UP", 2), ("RIGHT", 3)]`. If it can't read the text, it returns `None` and the game asks again.
* **`match_option`** accepts a list **number**, the **full name**, or a **unique part** of it (`mustard`, `2` or `Colonel Mustard`).
* **`validate_move`** is the rule checker. It walks a move one square at a time **without changing the game**, and rejects it (with the reason) if it:
  * Uses more steps than the roll
  * Leaves the board or hits a wall (including a room's wall)
  * Lands on another token's square
  * Re-enters the room the player just left
  * Stops short of the roll without entering a room

  Stepping onto a door puts the player **in the room and ends movement**, even with steps left over.

The cell below runs a few example moves for Miss Scarlett from her start square at (24, 7).

In [ ]:
from board_layout import BOARD_MAP, ROOM_CODES, START_POSITIONS, SECRET_PASSAGES
from board import Board
from input_utils import parse_move_command, validate_move

board = Board(BOARD_MAP, ROOM_CODES, START_POSITIONS, SECRET_PASSAGES)
scarlett = START_POSITIONS["Miss Scarlett"]

print("parse 'UP 2, right 3' ->", parse_move_command("UP 2, right 3"))
print("parse 'jump 3'        ->", parse_move_command("jump 3"))
print()
examples = [
  ("UP 3 with a roll of 3", scarlett, "UP 3", 3),
  ("LEFT 1 into the Lounge wall", scarlett, "LEFT 1", 1),
  ("UP 5 with a roll of 3", scarlett, "UP 5", 3),
  ("UP 1 with a roll of 3", scarlett, "UP 1", 3),
  ("DOWN 3 from (18, 5), the Lounge door is 1 step away", (18, 5), "DOWN 3", 6)
]
for label, start, command, roll in examples:
  result = validate_move(board, start, parse_move_command(command), roll, occupied=set())
  verdict = "OK " if result.ok else "NO "
  print(f"{verdict} {label:<52} {result.message}")

## 5. Players: `player.py`

Every player is a **`Player`** object. It stores:
* Their **character**
* Their **hand** of cards
* The cards other players have **shown** them (and who showed each one)
* Whether another player's suggestion **moved them** into a room. That gives them the classic `STAY` option on their next turn

**`HumanPlayer`** makes every decision by **prompting at the keyboard**:
* **`ask()`** handles the commands that work at every prompt: `BOARD`, `NOTES`, `HAND` and `HELP`.
* **`choose_move()`** first lists the **rooms you can reach** with your roll and the exact commands to get there.
* **`choose_suggestion()`** asks for a character and a weapon. The room is always the one you're standing in.
* **`choose_card_to_show()`** lets a refuting player pick which card to show when they hold more than one match.

The game engine only ever calls these `choose_...` methods. Decisions live in the player class and rules live in the engine, so the rules are in one place.

In [ ]:
from player import Player, HumanPlayer

decisions = [name for name in dir(HumanPlayer) if name.startswith("choose_") or name == "wait_for_roll"]
print("HumanPlayer decision methods:", ", ".join(decisions))
print("Every-prompt commands: BOARD, NOTES, HAND, HELP, QUIT")

## 6. The game engine: `game.py`

**`GameSetup`** builds a new game:
1. It checks the player list: 2–6 players, no character chosen twice.
2. It sorts players into the **standard turn order** (Miss Scarlett, then Colonel Mustard, and so on).
3. It places **all six character tokens** on their start squares, *including characters nobody is playing*, because a suggestion can still pull them into a room.
4. It puts the **weapons** in six different random rooms.
5. **Solution selection** (`_select_solution`): one random card from *each* category goes into the envelope and is never shown during play.
6. **Dealing** (`_deal_cards`): the other 18 cards are shuffled and dealt one at a time around the table. If they don't divide evenly, the first players get one extra card.

A token's location is a **room name** (inside that room) or a **`(row, col)` square** (in a hallway).

**`CluedoGame`** inherits from `GameSetup` and adds the rules of a turn:
* **`play()`:** players take turns in a fixed order until someone types `QUIT`. Then the envelope is revealed.
* **Movement from a hallway:** press Enter to **roll one die**, then type a move. Invalid moves are explained and asked again. `PASS` keeps you where you are.
* **Movement from a room:** you **can't just walk out**. Your choices are:
  * `ROLL`: roll, choose a door, step out (that uses **1** step), then spend the rest of the roll. You can't re-enter that room on the same turn.
  * `SECRET_PASSAGE_TO_…`: move **with no roll**, then make a suggestion.
  * `STAY`: only offered if another player's suggestion moved you there. You suggest without moving.
  * `PASS`: stay and do nothing.
* **Suggestions** are **required** whenever you *enter* a room, by walking in or by secret passage. You name a character and a weapon, and the room is the one you're in. The suggested character's token and the weapon are moved into the room immediately.
* **Refutation** starts with the **next player in turn order**. The **first** player holding any of the three cards shows **exactly one**, and they choose which if they hold several. Everyone sees *that* a card was shown. The card itself appears only in a line marked `[Message for <suggester>]`.

The cell below sets up a sample game with a fixed `seed`, so the dice and shuffles repeat exactly. It doesn't show anyone's cards or the envelope.

In [ ]:
from game import CluedoGame

sample = CluedoGame(["Mrs. Peacock", "Miss Scarlett", "Colonel Mustard", "Professor Plum"], seed=42)
print("Turn order:", " -> ".join(p.character for p in sample.players))
print("Cards dealt:", ", ".join(f"{p.character} {len(p.hand)}" for p in sample.players))
print("Weapon rooms:", ", ".join(f"{w} in the {r}" for w, r in sample.weapon_location.items()))
print("Envelope:", "sealed (", len(sample.solution), "cards )")

## 7. Self-tests: `tests.py`

These checks confirm the core rules behave as the brief describes. Each `assert` stops with an error if a rule is broken. They cover:
* **Move parsing:** valid and invalid command text.
* **Move validation:** walls, room walls, the board edge, too many steps, the exact-roll rule, blocked squares, entering a room ending the move, and no re-entering the room just left.
* **Setup:** one envelope card per category, 18 different dealt cards with none in the envelope, hand sizes within 1 of each other, and weapons in different rooms.
* **Suggestions:** the suggested character and weapon tokens move into the room, and the summoned player gets `STAY`.
* **Refutation order:** the first player to the suggester's left who holds a match is the one who shows a card.

The same tests run from a terminal with `python tests.py`.

In [ ]:
from tests import run_self_tests

run_self_tests()

## 8. Play the game

Run this cell and answer the prompts in the input box that appears under it. It calls the same `main()` that `python main.py` runs.

**Quick guide**
* **Setup example:** `3` players → `1` (Miss Scarlett) → `2` (Colonel Mustard) → `3` (Mrs. White)
* **Roll:** press **Enter**.
* **Move:** `UP 2, RIGHT 3` or `U2 R3`. The game lists the rooms you can reach and the exact command for each.
* **In a room:** `ROLL`, `SECRET_PASSAGE_TO_KITCHEN` (where there is one), or `PASS`.
* **Suggest:** `Mustard, Rope` or `2, 5`.
* **Anytime:** `BOARD`, `NOTES`, `HAND`, `HELP`, `QUIT` (quitting ends the game and reveals the envelope).

In [ ]:
from main import main

main()

## 9. Deliverables

**Folder Structure** :
```
ShahVivek_Project2_SourceCode/
├── src/
│   ├── main.py              <- entry point
│   ├── config.py
│   ├── cards.py
│   ├── board_layout.py
│   ├── board.py
│   ├── input_utils.py
│   ├── player.py
│   ├── game.py
│   └── tests.py
├── Colab Demo/
│   └── ShahVivek_Project2Part1_Demo.ipynb   <- this guided runner
└── ShahVivek_README.md
```

**Viewing the code:** open any `.py` file from Colab's **Files** panel, or run `%pycat game.py` in a cell (after the Setup cell has run).